# DSCI 552 Homework 3

-Name: Shiyi Fan  
-GitHub Username: 1111v1y\
-USC ID: 5554653109

## 1. Time Series Classification Part 1: Feature Creation/Extraction
### 1.(a)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
data3 = Path("../data/AReM/AReM")

### 1.(b)

In [3]:
anames = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]
def get_number(file_path):
    return int(file_path.stem.replace("dataset", ""))
file_records = []
for a in anames:
    adir = data3 / a
    files = sorted(
        adir.glob("dataset*.csv"),
        key=get_number
    )
    for file_path in files:
        datanumber = get_number(file_path)
        if a in ["bending1", "bending2"]:
            split = "test" if datanumber in [1, 2] else "train"
        else:
            split = "test" if datanumber in [1, 2, 3] else "train"
        file_records.append({
            "activity": a,
            "dataset": datanumber,
            "file_path": file_path,
            "split": split
        })
split_df = pd.DataFrame(file_records)
pd.crosstab(
    split_df["activity"],
    split_df["split"],
    margins=True
)

split,test,train,All
activity,,,
bending1,2,5,7
bending2,2,4,6
cycling,3,12,15
lying,3,12,15
sitting,3,12,15
standing,3,12,15
walking,3,12,15
All,19,69,88


### 1.(c)
#### 1.(c)(i)

##### -minimum, maximum, mean, median, standard deviation, variance, first quartile, third quartile, range, interquartile range, skewness, kurtosis, root mean square

#### 1.(c)(ii)

In [4]:
cnames = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "ar_rss23"
]
def read_dataset(file_path):
    try:
        data = pd.read_csv(
            file_path,
            skiprows=5,
            header=None
        )
    except pd.errors.ParserError:
        data = pd.read_csv(
            file_path,
            skiprows=5,
            header=None,
            usecols=range(7)
        )
    if data.shape[1] != 7:
        data = pd.read_csv(
            file_path,
            skiprows=5,
            header=None,
            sep=r"\s+"
        )
    data.columns = cnames
    return data.apply(pd.to_numeric)
feature_rows = []
for file_path in split_df["file_path"]:
    data = read_dataset(file_path)
    rfeatures = {}
    for i, column in enumerate(cnames[1:], start=1):
        series = data[column]
        rfeatures[f"min{i}"] = series.min()
        rfeatures[f"max{i}"] = series.max()
        rfeatures[f"mean{i}"] = series.mean()
        rfeatures[f"median{i}"] = series.median()
        rfeatures[f"std{i}"] = series.std()
        rfeatures[f"1st quart{i}"] = series.quantile(0.25)
        rfeatures[f"3rd quart{i}"] = series.quantile(0.75)
    feature_rows.append(rfeatures)

feature_df = pd.DataFrame(feature_rows)
feature_df.index = range(1, len(feature_df) + 1)
feature_df.index.name = "Instance"
feature_df

,min1,max1,mean1,median1,std1,1st quart1,3rd quart1,min2,max2,mean2,...,std5,1st quart5,3rd quart5,min6,max6,mean6,median6,std6,1st quart6,3rd quart6
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,4.576562,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,4.456333,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,4.371958,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350


#### 1.(c)(iii)

In [5]:
rng = np.random.default_rng(42)
bootstrap_results = []
for feature in feature_df.columns:
    values = feature_df[feature].to_numpy()
    std_estimate = np.std(values, ddof=1)
    bootstrap_stds = []
    for _ in range(10000):
        bootstrap_sample = rng.choice(
            values,
            size=len(values),
            replace=True
        )
        bootstrap_stds.append(
            np.std(bootstrap_sample, ddof=1)
        )
    cilower, ciupper = np.percentile(
        bootstrap_stds,
        [5, 95]
    )
    bootstrap_results.append({
        "Feature": feature,
        "Standard Deviation": std_estimate,
        "90% CI Lower": cilower,
        "90% CI Upper": ciupper
    })

bootstrap_df = pd.DataFrame(bootstrap_results)
bootstrap_df.set_index("Feature", inplace=True)
bootstrap_df

,Standard Deviation,90% CI Lower,90% CI Upper
Feature,,,
min1,9.569975,8.236003,10.773895
max1,4.394362,3.329945,5.296590
mean1,5.335718,4.688296,5.889086
median1,5.440054,4.795819,5.987308
std1,1.772153,1.574247,1.951729
1st quart1,6.153590,5.575910,6.643478
3rd quart1,5.138925,4.326313,5.842674
min2,0.000000,0.000000,0.000000
max2,5.062729,4.631065,5.399928


#### 1.(c)(iv)

##### choose min, max and mean
##### Min describe the lowest value of TS; Max describe the highest value of TS; Mean describe the average level.

### 3.7.4

#### (a)

##### Cubic regression has more terms, so it can fit the training data at least as well as a linear model.

#### (b)

##### Linear regression. The additional terms in the cubic model may fit the noise, leading to overfitting.

#### (c)

##### Cubic regression is more flexible rather than linear regression, so it can fit the training data better.

#### (d)

##### I am not sure. Cubic model may perform better if the true relationship is close to cubic, otherwise linear model may perform better.